<a href="https://colab.research.google.com/github/pySTEPS/ERAD-nowcasting-course-2026/blob/main/notebooks/exercise_notebooks/block_05_blending.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

These files contain the solutions of the exercises. For the exercise notebooks, go to [exercise_notebooks](https://github.com/pySTEPS/ERAD-nowcasting-course-2026/tree/main/notebooks/exercise_notebooks).

# Blending with NWP

In this final notebook, we blend a radar rainfall nowcast with an NWP rainfall forecast, using two of the blending methods in pysteps: [linear blending](https://pysteps.readthedocs.io/en/latest/auto_examples/plot_linear_blending.html) (including its saliency-based variant) and [STEPS blending](https://pysteps.readthedocs.io/en/latest/auto_examples/blended_forecast.html).

A nowcast is skilful for the first hour or so. After this period, the extrapolatin assumptions start to break down due to the dynamical evolution of the atmosphere. A numerical weather forecast (NWP) is less similar to the latest observations for short lead times, but better describes the full atmospheric evolution than an extrapolation-based nowcast, providing more skill at longer lead times. Blending combines the two, giving more weight to the extrapolation nowcast at short lead times and the NWP at longer ones. Such a combined approach which aims to optimize skill at all lead times is also referred to as seamless nowcasting or **seamless prediction**.

We use the Serbian case of 12 August 2017: the RHMSS radar composite and the ECMWF IFS control run initialised at midnight 00:00 UTC. The blended nowcast is issued at **15:55 UTC** and runs **2 hours** ahead.

## The data

We use the RHMSS radar composite of 12 August 2017 and, as NWP forecast, the last ECMWF IFS control run available before the nowcast: the one initialised at 00 UTC that morning.

The radar composite is read by [helper_input_data.ipynb](helper_input_data.ipynb), just like in the previous blocks. It gives us the composite as `precip` (in mm/h) and its `metadata`.

In [ ]:
from google.colab import drive
import os
# mount the Google Drive folder
# don't attempt to remount if the drive is already mounted
if not os.path.exists("/content/mnt/MyDrive"):
  drive.mount("mnt")
%cd '/content/mnt/MyDrive/Colab Notebooks/ERAD-nowcasting-course-2026/notebooks/exercise_notebooks/'
# run the data notebook to configure the environment and load the radar data
%run helper_input_data.ipynb

The IFS forecast comes in a file format that pysteps itself does not read. Importers for such data can be added as plugins, and the [pysteps-nwp-importers](https://github.com/pySTEPS/pysteps-nwp-importers) plugin provides the one for this dataset. 

In [ ]:
# The NWP forecasts are GRIB files, read through the pysteps-nwp-importers plugin
!apt-get install -qq libeccodes-dev
!uv pip install pygrib
!uv pip install git+https://github.com/pySTEPS/pysteps-nwp-importers@add_importer_rhmss_nwp

With the plugin installed, we can read the IFS forecast. It contains hourly precipitation accumulations clipped for a domain over Serbia. We we convert the accumulations to mean rain rates, so that both sources are in mm/h.

In [ ]:
import pysteps
from pysteps_nwp_importers.importer_rhmss_nwp import import_rhmss_nwp

nwp_precip_native, _, nwp_metadata_native = import_rhmss_nwp(
    "/content/mnt/MyDrive/Colab Notebooks/erad_data/ifs_20170812control.nc"
)

# Convert the hourly accumulations (mm) to mean rain rates (mm/h)
converter = pysteps.utils.get_method("mm/h")
nwp_precip_native, nwp_metadata_native = converter(nwp_precip_native, nwp_metadata_native)
print("IFS forecast:", nwp_precip_native.shape,
      "|", nwp_metadata_native["accutime"], "min accumulations")

The helper notebook takes it from here: it splits the radar composite into nowcast input and observations, puts the NWP forecast on the radar grid and on the 5-minute nowcast time steps, and does the pre-processing — see [helper_blending.ipynb](helper_blending.ipynb) for the details.

In [ ]:
# run the helper notebook to prepare the radar and NWP data for the blending
%run helper_blending.ipynb

## Inspecting the NWP data

The IFS runs on a much coarser grid than the radar observations, and the two products differ noticeably. The left panel shows the forecast on its own grid, the middle one after the helper reprojected it onto the radar grid, and the right panel compares to observation

In [ ]:
# Disable warnings
import warnings
warnings.filterwarnings("ignore")

from matplotlib import pyplot as plt
from pysteps.visualization import plot_precip_field

map_kwargs = {"drawlonlatlines": True}
date_str = f"{date_radar:%Y-%m-%d %H:%M}"
i_native = int(np.searchsorted(nwp_metadata_native["timestamps"], date_radar, side="left"))
# derive date for the NWP field
date_nwp = nwp_metadata_native["timestamps"][i_native]

plt.figure(figsize=(16, 5))
panels = [
          (nwp_precip_native[i_native], nwp_metadata_native, f"IFS on its own grid at {date_nwp}"),
          (nwp_precip[0], nwp_metadata, f"IFS on the radar grid at {date_nwp}"),
          (radar_precip[-1], radar_metadata, f"Radar observation at {date_str}")]
          
for i, (field, geo, title) in enumerate(panels):
    plt.subplot(1, 3, i + 1).set_axis_off()
    plot_precip_field(field, geodata=geo, colorscale="pysteps", title=title,
                      colorbar=(i == 2), map_kwargs=map_kwargs)
plt.tight_layout()
plt.show()

## A helper to look at the results

Every blending method below is inspected in the same way, so we define the plotting and verification once. We use the CRPS as skill score, because it works for both ensemble and deterministic forecasts (in the latter case, it reduces to the mean absolute error or MAE).

In [ ]:
from pysteps.verification import probscores

leadtimes_min = [5, 30, 60, 90, 120]

def plot_and_verify(forecast, label):
    """Plot a few lead times against the observations, and plot the CRPS."""
    # accept both deterministic (time, y, x) and ensemble (member, time, y, x)
    ens = forecast if forecast.ndim == 4 else forecast[None]

    plt.figure(figsize=(11, 2.6 * len(leadtimes_min)))
    for n, lt in enumerate(leadtimes_min):
        i = int(lt / timestep) - 1
        panels = [(ens[:, i].mean(axis=0), f"{label} +{lt} min"),
                  (nwp_precip[i], f"IFS +{lt} min"),
                  (precip_obs[i], f"Observation +{lt} min")]
        for col, (field, title) in enumerate(panels):
            plt.subplot(len(leadtimes_min), 3, n * 3 + col + 1).set_axis_off()
            plot_precip_field(field, geodata=radar_metadata, title=title,
                              axis="off", colorbar=False)
    plt.tight_layout()
    plt.show()

    crps_blended, crps_nwp = [], []
    for i in range(n_nowcast_steps):
        crps_blended.append(probscores.CRPS(ens[:, i], precip_obs[i]))
        crps_nwp.append(probscores.CRPS(nwp_precip[i][None], precip_obs[i]))

    fig, ax = plt.subplots(figsize=(9, 4))
    lts = (np.arange(n_nowcast_steps) + 1) * timestep
    ax.plot(lts, crps_nwp, color="tab:blue", lw=2, label="IFS")
    ax.plot(lts, crps_blended, color="tab:orange", lw=2, label=label)
    ax.set_xlabel("Lead time (min)", fontsize=12)
    ax.set_ylabel(r"CRPS (mm h$^{-1}$)", fontsize=12)
    ax.set_title(f"CRPS for the forecast issued at {date_radar:%Y-%m-%d %H:%M} UTC")
    ax.legend(frameon=False, fontsize=12)
    plt.tight_layout()
    plt.show()
    return crps_blended

## Linear blending

The simplest approach: give the nowcast all the weight until `start_blending`, the NWP all the weight from `end_blending` onwards, and interpolate linearly in between. Both times are chosen by the user.

In [ ]:
start_blending = 15
end_blending = 45
precip_blended_linear = pysteps.blending.linear_blending.forecast(
    precip=radar_precip_db[-1, :, :],
    precip_metadata=radar_metadata_db,
    velocity=velocity_radar,
    timesteps=n_nowcast_steps,
    timestep=timestep,
    nowcast_method="extrapolation",   # simple advection nowcast
    start_blending=15,   # in minutes
    end_blending=45,     # in minutes
    precip_nwp=nwp_precip,
    precip_nwp_metadata=nwp_metadata,
)

_ = plot_and_verify(precip_blended_linear, "Linear blending")

## Saliency-based blending

Plain linear blending smooths away the intense cells during the transition. The saliency-based variant preserves pixel intensities that stand out from their surroundings, by ranking them before combining the two forecasts. It is the same function with `saliency=True`.

In [ ]:
precip_blended_salient = pysteps.blending.linear_blending.forecast(
    precip=radar_precip_db[-1, :, :],
    precip_metadata=radar_metadata_db,
    velocity=velocity_radar,
    timesteps=n_nowcast_steps,
    timestep=timestep,
    nowcast_method="extrapolation",
    start_blending=15,
    end_blending=45,
    precip_nwp=nwp_precip,
    precip_nwp_metadata=nwp_metadata,
    saliency=True,
)

_ = plot_and_verify(precip_blended_salient, "Salient blending")

## STEPS blending

Both methods above need the user to pick when the blending starts and ends. The [STEPS blending method](https://pysteps.readthedocs.io/en/latest/pysteps_reference/blending.html) instead derives the weights per spatial scale from the skill of each component, so the transition happens where the data says it should. It also perturbs both components, giving an ensemble.

It needs the NWP forecast in dB, with a leading model dimension, and a motion field for the NWP.

In [ ]:
# Transform the NWP forecast to dB and add the model dimension
nwp_precip_db, nwp_metadata_db = transformer(nwp_precip_steps, nwp_metadata, threshold=0.1)
nwp_precip_db = nwp_precip_db[None, :]

# The motion field of the NWP forecast, one per model and time step
velocity_nwp = []
for n_model in range(nwp_precip_db.shape[0]):
    v = [oflow_method(nwp_precip_db[n_model, t - 1:t + 1])
         for t in range(1, nwp_precip_db.shape[1])]
    # the field at the first time step is the same as at the second
    velocity_nwp.append(np.insert(v, 0, v[0], axis=0))
velocity_nwp = np.stack(velocity_nwp)

precip_blended_steps = pysteps.blending.steps.forecast(
    precip=radar_precip_db,
    precip_models=nwp_precip_db,
    velocity=velocity_radar,
    velocity_models=velocity_nwp,
    timesteps=n_nowcast_steps,
    timestep=timestep,
    issuetime=date_radar,
    n_ens_members=1,     # keep it to 1 member so the notebook stays quick
    precip_thr=radar_metadata_db["threshold"],
    kmperpixel=radar_metadata["xpixelsize"] / 1000.0,
    noise_stddev_adj="auto",
    vel_pert_method=None,
)

# Back to mm/h before verifying
precip_blended_steps, _ = converter(precip_blended_steps, radar_metadata_db)

_ = plot_and_verify(precip_blended_steps, "STEPS blending")

## Things to try or think about

* Move `start_blending` and `end_blending`. The values above were picked by comparing a pure extrapolation nowcast with the IFS lead time by lead time: the nowcast stops winning at +20 min, so the blending is centred there. Can you find a better pair? What happens if you blend  much later, say 60 and 120 minutes?
* Replace `nowcast_method="extrapolation"` with `"steps"` in the linear blending, and pass `nowcast_kwargs={"precip_thr": 0.1, "kmperpixel": 1.0, "timestep": 5, "n_ens_members": 10}`. Does an ensemble nowcast improve the CRPS?
* Raise `n_ens_members` in the STEPS blending (it costs run time, but the CRPS of a single member is pessimistic).
* Is there anything unusual you notice about the STEPS blending? Something to keep in mind is that the motion vectors also evolve towards the optical flow derived from the NWP fields, and these are hourly aggregated fields.
* What happens if you blend with the WRF-NMM forecast (`NMM20170812.nc`) instead of the IFS? It is on a finer grid, but only available as 3-hourly accumulations — what would that do to the blended forecast?